In [4]:
import os
import sys
import time
import sqlite3
import pandas as pd
from datetime import datetime

# パス設定（jupyter/py 両対応）
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config import PROJECT_DIR

start_time = time.time()

# DBパス設定
user_base = os.path.join(
    os.environ["USERPROFILE"] if os.name == 'nt' else os.path.expanduser("~"),
    "myenv310", PROJECT_DIR
)
db_path = os.path.join(user_base, "db", "output.db")
print(f"[INFO] 使用DB: {db_path}")

# 種別履歴カラムを取得（直近から10回前まで）
history_cols = [f"ステータス{i}回前" for i in range(1, 11)]
columns = ["実行日", "台番号"] + history_cols

# データ取得
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query(f'''
        SELECT {", ".join([f"[{col}]" for col in columns])}
        FROM result_table
        ORDER BY ROWID DESC
    ''', conn)

# 実行日で最新データに絞る
df["実行日"] = pd.to_datetime(df["実行日"], errors='coerce')
df = df.dropna(subset=["実行日"])
max_date = df["実行日"].dt.date.max()
df_today = df[df["実行日"].dt.date == max_date].copy()
print(f"[INFO] 最新実行日: {max_date}, 件数: {len(df_today)}")

# ---------- bb駆け抜け判定関数----------
def judge_bb_escape(row):
    history = [str(row.get(f"ステータス{i}回前", "")).strip() for i in range(1, 11)]

    for i in range(len(history) - 1):
        curr = history[i]       # 新しい（1回前, 2回前, …）
        prev = history[i + 1]   # 1つ古い

        # BIG → REG
        if prev == "BIG" and curr == "REG":
            return 1   # 成立（単発/連チャン関係なく成立とみなす）

        # BIG → AT/ART
        if prev == "BIG" and curr == "AT/ART":
            return 1   # 成立（単発/連チャン関係なく成立とみなす）

    return None


# ---------- 判定列作成 ----------
df_today["BB駆け抜け判定"] = df_today.apply(judge_bb_escape, axis=1)

# 結果表示
count_escape = df_today["BB駆け抜け判定"].notna().sum()
print(f"[INFO] BB駆け抜け対象: {count_escape} 件")

# ---------- DB更新 ----------
with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()

    update_sql = '''
        UPDATE result_table
        SET [BB駆け抜け判定] = ?
        WHERE [台番号] = ? AND date([実行日]) = ?
    '''

    updated = 0
    for _, row in df_today.iterrows():
        cur.execute(update_sql, (
            row.get("BB駆け抜け判定"),
            row["台番号"],
            row["実行日"].strftime('%Y-%m-%d')
        ))
        updated += 1

    conn.commit()

print(f"✅ BB駆け抜け判定 更新完了: {updated}件")
print(f"[INFO] 所要時間: {time.time() - start_time:.1f}秒")


[INFO] 使用DB: C:\Users\stray\myenv310\friend3-s\db\output.db
[INFO] 最新実行日: 2025-08-20, 件数: 170
[INFO] BB駆け抜け対象: 59 件
✅ BB駆け抜け判定 更新完了: 170件
[INFO] 所要時間: 0.0秒
